# 14 协程

> 本笔记围绕 Python `asyncio` 协程展开，内容包括：协程的基本概念、协程函数与协程对象、`await` 关键字、同步执行与异步执行、多任务调度、`asyncio.gather`，以及图片下载实战案例。

---

## 学习目标

学完本笔记后，你应该能够理解：

1. 什么是协程，以及它和线程、进程的区别。
2. 什么是协程函数、协程对象。
3. `await` 为什么能实现“挂起与恢复”。
4. 为什么 `asyncio.create_task()` 能实现并发调度。
5. `asyncio.gather()` 的典型用法。
6. 如何使用协程完成 IO 密集型任务，例如并发下载图片。


## 1. 什么是协程

### 概念

协程（Coroutine），是一种**线程内部的任务调度机制**。它通过**事件循环**，在**用户态**中实现任务的挂起与恢复执行，从而在遇到 IO 操作时，不让 CPU 空等，而是继续执行其它需要 CPU 的任务。

协程的本质就是：

> 在一个线程里，趁着某些任务在等 IO，把 CPU 交给其它任务去用。

### 关键点 1：协程不是线程，也不是进程

- 协程不是操作系统提供的。
- CPU 看不见协程。
- 操作系统不知道协程的存在。
- 协程是程序员在用户态，用代码设计出来的一套任务切换机制。

### 关键点 2：协程发生在一个线程内部

- 协程不是线程之间的切换。
- 而是**一个线程内部多个任务之间的切换**。
- 本质是：一个线程里写了很多任务，由事件循环统一调度。

### 关键点 3：协程的核心能力是挂起与恢复

- 当任务遇到 **IO 操作** 时：任务会被挂起。
- 当 IO 操作完成后：任务会被恢复执行。

### 关键点 4：协程依赖事件循环

事件循环负责：

- 调度任务
- 判断是否该挂起
- 决定何时恢复执行

可以把事件循环理解为协程系统的“大脑”。

### 关键点 5：协程的目标是尽量减少线程切换

在单线程场景下，协程的目标是尽可能提高 CPU 利用率，尤其适合：

- 网络请求
- 文件读写
- 数据库访问
- 爬虫抓取

这类场景有一个共同点：**IO 等待时间长，CPU 真正计算时间短**。


### 协程、线程、进程的简单对比

| 对比项 | 进程 | 线程 | 协程 |
|---|---|---|---|
| 是否由操作系统直接管理 | 是 | 是 | 否 |
| 切换位置 | 进程之间 | 线程之间 | 同一线程内部 |
| 切换成本 | 高 | 较高 | 很低 |
| 是否适合 IO 密集型 | 可以 | 可以 | 非常适合 |
| 是否能利用多核 | 可以 | 可以 | 单个协程本身不直接利用多核 |

### 一个形象化理解

- **进程**：像一栋栋独立的房子。
- **线程**：像房子里的不同房间，可以并行做事。
- **协程**：像同一个房间里，一个人手里安排的多项事务，遇到等待时就先去做别的。


## 2. 协程函数 vs 协程对象

### 协程函数（coroutine function）

使用 `async` 关键字修饰的函数，就是协程函数。

例如：

- `async def work(): ...`

### 协程对象（coroutine object）

调用协程函数后，得到的不是立即执行的结果，而是一个**协程对象**。

### 特别注意

> 调用协程函数，并不会立刻执行函数体中的代码。

协程对象必须被交给事件循环，才能真正开始执行。


In [ ]:
import asyncio

# 定义一个协程函数
async def work():
    print('work开始')
    print('work执行中......')
    print('work结束')
    return '工作结果'

# 调用协程函数，会得到协程对象
coroutine_object = work()
print(coroutine_object)

# 将协程对象交给 asyncio.run() 执行
result = asyncio.run(coroutine_object)
print(result)


### `asyncio.run()` 做了什么？

`asyncio.run()` 可以看作协程程序的入口，它通常会做 3 件事：

1. 创建一个事件循环。
2. 把收到的协程对象包装成任务并交给事件循环。
3. 启动事件循环，直到任务执行结束。

### 返回值说明

`asyncio.run()` 会阻塞当前线程，直到协程执行结束，并返回该协程的 `return` 结果。


## 3. `await` 关键字

`await` 是协程中最核心的关键字之一，它主要有三个作用：

### 1）挂起

`await` 会暂停当前协程的执行。

### 2）等待

遇到 `await` 后，事件循环会安排 `await` 后面的对象去执行，并等待其完成，同时还能拿到执行结果。

### 3）恢复

当 `await` 后面的对象执行结束后，事件循环会恢复之前被挂起的协程，继续往下执行。

---

## `await` 后到底发生了什么？

执行 `await` 后面的对象时，通常会出现两种情况：

### 情况一：对象内部发生了 IO 等待

例如：

- 网络请求
- 文件读写
- 数据库查询

如果在执行过程中遇到了这类等待，当前协程就会把 CPU 控制权交还给事件循环，事件循环可以趁机调度其它任务。

### 情况二：对象内部没有任何 IO 等待

例如：

- `print`
- 数学运算
- 逻辑判断

此时虽然写了 `await`，但如果执行的内容本身并没有真正让出控制权，那么事件循环也很难切换到其它任务。

### 注意事项

> `await` 后面只能写“可等待对象”。

常见可等待对象包括：

- 协程对象
- `Task` 对象
- `Future` 对象


In [ ]:
import asyncio

async def work():
    print('work开始')
    print('work执行中......')
    # asyncio.sleep() 返回的是一个协程对象
    res = await asyncio.sleep(2)
    print(res)
    print('work结束')
    return '工作结果'

async def main():
    print('main开始')
    # await 去等待一个协程对象
    res = await work()
    print(res)
    print('main结束')
    return 'main的返回值'

result = asyncio.run(main())
print(result)


## 4. 多个任务同步执行

虽然协程具备“可挂起”的能力，但如果你在代码中是一个接一个地 `await`，那么整体执行效果仍然可能是**同步的**。

也就是说：

- 先等任务 1 完成
- 再等任务 2 完成
- 再等任务 3 完成

这种写法下，后一个任务必须等前一个任务结束后才会开始。


In [ ]:
import asyncio
import time

# 定义一个协程函数
async def work(n, delay):
    print(f'work{n}开始')
    print(f'work{n}执行中......')
    # 模拟一个 IO 等待
    await asyncio.sleep(delay)
    print(f'work{n}结束')
    return f'work{n}的返回值'

async def main():
    print('main开始')
    start = time.time()

    # 调用三次 work 函数，分别得到三个协程对象
    coroutine1 = work(1, 2)
    coroutine2 = work(2, 2)
    coroutine3 = work(3, 2)

    # 顺序等待，所以整体表现为同步执行
    res1 = await coroutine1
    print(res1)

    res2 = await coroutine2
    print(res2)

    res3 = await coroutine3
    print(res3)

    print('main结束', time.time() - start)
    return '我是main的返回值'

result = asyncio.run(main())
print(result)


### 为什么这是同步执行？

因为虽然写的是协程函数，但你并没有把这 3 个任务同时交给事件循环调度，而是：

- 先 `await coroutine1`
- 等 `coroutine1` 完成后
- 再 `await coroutine2`
- 再等 `coroutine2` 完成后
- 最后 `await coroutine3`

因此总耗时大约是：

`2 秒 + 2 秒 + 2 秒 = 6 秒左右`


## 5. 多个任务异步执行

如果希望多个任务能够被事件循环**并发调度**，需要显式把它们注册为任务。

最常用的方法是：

- `asyncio.create_task()`

这个方法会把协程对象包装成一个可调度的 `Task`，并立即注册到事件循环中。

这样多个任务就有机会交替执行。


In [ ]:
import asyncio
import time

async def work(n, delay):
    print(f'work{n}开始')
    print(f'work{n}执行中......')
    # 模拟一个 IO 等待
    await asyncio.sleep(delay)
    print(f'work{n}结束')
    return f'work{n}的返回值'

async def main():
    print('main开始')
    start = time.time()

    # create_task 会把协程对象包装成任务并注册到事件循环
    task1 = asyncio.create_task(work(1, 2))
    task2 = asyncio.create_task(work(2, 2))
    task3 = asyncio.create_task(work(3, 2))

    # 虽然这里依旧按顺序 await，但任务已经提前开始运行了
    res1 = await task1
    print(res1)

    res2 = await task2
    print(res2)

    res3 = await task3
    print(res3)

    print('main结束', time.time() - start)
    return '我是main的返回值'

result = asyncio.run(main())
print(result)


### 这里为什么会变快？

因为 3 个任务都已经被加入事件循环中了。

执行流程可以理解为：

1. `task1` 启动，遇到 `await asyncio.sleep(2)` 后挂起。
2. 事件循环去执行 `task2`，它也很快挂起。
3. 事件循环再去执行 `task3`，它也挂起。
4. 之后谁先等到 IO 完成，谁就恢复执行。

由于这三个任务的等待时间都是 2 秒，所以总耗时通常接近 **2 秒**，而不是 6 秒。


## 6. `asyncio.gather`

`asyncio.gather()` 是协程编程中非常常用的一个工具。

它的作用是：

> 把多个可等待对象一起交给事件循环，并在它们全部完成后，一次性收集结果。

它特别适合：

- 同时发多个请求
- 同时启动多个下载任务
- 同时执行多个彼此独立的 IO 任务


In [ ]:
import asyncio
import time

async def work(n, delay):
    print(f'work{n}开始')
    print(f'work{n}执行中......')
    # 模拟一个 IO 等待
    await asyncio.sleep(delay)
    print(f'work{n}结束')
    return f'work{n}的返回值'

async def main():
    print('main开始')
    start = time.time()

    # 把多个协程对象同时交给事件循环
    result = await asyncio.gather(
        work(1, 2),
        work(2, 2),
        work(3, 2)
    )

    print(result)
    print('main结束', time.time() - start)
    return '我是main的返回值'

result = asyncio.run(main())
print(result)


### `gather` 的特点

1. 可以一次性提交多个任务。
2. 会等待所有任务都完成。
3. 返回结果是一个列表，顺序与传入顺序一致。

例如上面的返回值可能是：

```python
['work1的返回值', 'work2的返回值', 'work3的返回值']
```

### `gather` 与 `create_task` 的关系

- `create_task()` 更偏向“先创建任务，再按需要等待”。
- `gather()` 更偏向“把一组任务一起提交，并统一收集结果”。

在实际开发中，这两种方式都很常见。


## 7. 下载图片案例

下面通过一个实际案例，感受协程在 IO 密集型场景中的优势。

我们会分别展示：

1. 传统同步下载方式
2. 使用协程进行并发下载


### 7.1 使用传统方式下载图片

这种写法的特点是：

- 图片一张一张下载
- 当前图片没有下载完成，下一张图片不会开始
- 这是典型的同步下载


In [ ]:
import requests

def download_picture(url):
    print(f'开始下载：{url}')
    # 发送网络请求，获取图片
    response = requests.get(url)
    print('下载完毕')

    # 保存图片到本地
    with open(url[-10:], 'wb') as file:
        file.write(response.content)

def main():
    url_list = [
        'https://n.sinaimg.cn/spider20260129/217/w600h417/20260129/3e26-917ee55a8a42b8626807c332c24981de.png',
        'https://n.sinaimg.cn/finance/transform/97/w630h267/20260129/97c4-b211cc51784830f09ee19e450475c93b.png',
        'https://n.sinaimg.cn/spider20260129/539/w1439h700/20260129/e09a-cc2ca319e00f701ccfca3ebc62aa8772.png'
    ]

    for url in url_list:
        download_picture(url)

main()


### 7.2 使用协程方式下载图片

这种写法的特点是：

- 多张图片会几乎同时发起请求
- 某一张图片在等待网络返回时，其它任务不会被阻塞
- 这就是协程在 IO 密集型任务中的典型优势

这里我们使用 `aiohttp` 配合 `asyncio` 完成异步下载。


In [ ]:
import aiohttp
import asyncio

async def download_picture(session, url):
    print(f'开始下载：{url}')

    # 发送请求：请求发出去后，需要等待服务器返回响应
    response = await session.get(url)

    # 读取响应体：图片数据可能会分多次传输，也需要等待
    content = await response.read()
    print('下载完毕')

    # 保存图片到本地
    with open(url[-10:], 'wb') as file:
        file.write(content)

    # 释放连接资源
    response.release()

async def main():
    url_list = [
        'https://n.sinaimg.cn/spider20260129/217/w600h417/20260129/3e26-917ee55a8a42b8626807c332c24981de.png',
        'https://n.sinaimg.cn/finance/transform/97/w630h267/20260129/97c4-b211cc51784830f09ee19e450475c93b.png',
        'https://n.sinaimg.cn/spider20260129/539/w1439h700/20260129/e09a-cc2ca319e00f701ccfca3ebc62aa8772.png'
    ]

    # 创建会话对象
    async with aiohttp.ClientSession() as session:
        # 创建多个协程对象
        coroutine_list = [download_picture(session, url) for url in url_list]

        # 同时交给事件循环执行
        await asyncio.gather(*coroutine_list)

asyncio.run(main())


### 为什么这里效率更高？

因为网络下载本质上是 IO 密集型任务。

在同步写法中：

- 程序请求一张图片后
- 只能傻等服务器返回
- 等完再请求下一张

而在协程写法中：

- 请求 1 发出去后，进入等待
- 事件循环立刻去发请求 2
- 请求 2 等待时，再去发请求 3
- 整个过程把“等待时间”充分利用起来了

这就是协程并发的核心价值。


## 8. 协程使用时的注意事项

### 1）协程更适合 IO 密集型任务

协程的优势来自“等待期间切换任务”。如果你的任务几乎全是计算，例如：

- 大量数学运算
- 图像计算
- 模型训练

那么协程带来的收益通常不大。

### 2）不要把阻塞代码直接塞进协程里

例如在协程内部直接调用：

- `time.sleep()`
- 阻塞式网络请求
- 阻塞式文件操作

这些操作会卡住整个线程，让事件循环无法调度其它任务。

### 3）`asyncio.sleep()` 和 `time.sleep()` 完全不同

- `time.sleep()`：阻塞线程
- `await asyncio.sleep()`：挂起当前协程，不阻塞整个事件循环

### 4）并发不等于并行

- **并发**：多个任务交替推进
- **并行**：多个任务同一时刻真正同时运行

单线程协程主要解决的是并发问题，而不是多核并行问题。


## 9. 总结

本节核心内容可以归纳为以下几点：

1. **协程是一种线程内部的调度机制**，适合处理 IO 密集型任务。
2. **协程函数**由 `async def` 定义，调用后得到的是**协程对象**。
3. `await` 的本质是：**挂起当前协程，等待目标完成，之后再恢复执行**。
4. 仅仅写协程函数并不代表任务会并发执行，关键在于是否交给事件循环统一调度。
5. `asyncio.create_task()` 可以把协程注册为任务，从而实现并发推进。
6. `asyncio.gather()` 能够一次性调度多个任务并收集结果。
7. 在图片下载、网络请求、文件读写等场景中，协程通常能显著提升效率。

---

## 一句话记忆

> 协程就是：在一个线程里，某个任务遇到 IO 等待时，先让出来，去执行别的任务，等条件满足后再回来继续执行。


## 10. 练习建议

为了真正掌握协程，建议继续练习以下题目：

1. 写 5 个协程任务，分别等待不同秒数，观察输出顺序。
2. 对比 `await coroutine`、`create_task()`、`gather()` 的效果差异。
3. 把同步网络请求改写成异步请求。
4. 尝试编写一个异步爬虫，抓取多个页面标题。
5. 尝试用协程同时读取多个文件内容。
